# Hyper Instancing Meshes in a BIM Model - 3

In part 2 of this notebook, we attempted to create an IfcMappedItem within our IFC file to reduce the number of unique counts of geometries for Pipes in the model. To recap, our goal is to replace all straight lengths of pipe in the model with one instanced "Unit Pipe", that can be rotated, translated or scaled accordingly. This allows for strict instancing within a viewing software, improving both memory and performance of the scene. This work is influenced by the research paper "The Unit Pipe: A Memory-Efficient Representation for Real-Time Visualization of Massive MEP Models", by Mikael Johansson and Mattias Roupé.

In Part 1, we were able to write a function to reproducibly extract the transformation matrix for a given pipe. This process entailed capturing the covariance matrix of the pipe, extracting the eigenvetors, and using these derived axes to find the scale of our components in each axis. This gave us a nice 4x4 matrix that captured the entire transformation operation of the pipe with great results.

Now, we shall extend this out to create a custom gltf file that contains only instanced pipes. In theory, this file should be a minute fraction of the total file size of our 3D model file, but with no meaningful difference in visuals. Theoretically spaeking, we could also control the per-instance attributes of the geometry through the use of Accessors and bufferviews. Let's get into it.

## Import of Modules and Packages

In [1]:
import numpy as np
import pygltflib
import pymeshlab
import ifcopenshell
import ifcopenshell.geom
import ifcopenshell.util.shape

Let's import our 3D model.

In [2]:
model = ifcopenshell.open('data/O-S1-INS-Plumbing - Sanitair.ifc')

Selecting only the pipes --

In [3]:
pipes = model.by_type("IfcFlowSegment")

And some settings for our geometry iterator.

In [4]:
settings = ifcopenshell.geom.settings()
geom_library="hybrid-cgal-simple-opencascade"

As well, lets import our unit pipe.

In [5]:
# Loading our unit pipe
ms = pymeshlab.MeshSet()
ms.load_new_mesh('data/simple_pipe.obj')

m = ms.current_mesh()
v_matrix = m.vertex_matrix()

f_matrix = m.face_matrix()

In [6]:
v_matrix = np.array(v_matrix, dtype="float32")
f_matrix = np.array(f_matrix, dtype='uint8')

Now that we have everything set up, we should be able to get to work.

## Proof of Concept

Before committing to our entire Ifc file, I'd like to ensure that stuff works. For this reason, we created a direct export of our IFC file from Blender to gltf file format, and we shall be using that as a baseline.

**Our Naive exported file is 42MB in size on disk. When loaded to the scene, it consumes 300MB of memory.** Its important to note that we have declined to export materials, to truly compare apples to apples here.

We have built manual GLTF files before, and shall be referencing the [work done in this notebook](../../optimizing-the-scene/notebooks/gltf-eda.ipynb).

We start by creating a blank gltf project.

In [7]:
# Initialize empty GLTF container to store the newly created mesh objects.
gltf_export = pygltflib.GLTF2()
gltf_export.scenes.append(pygltflib.Scene(nodes=[]))
gltf_export.scene = 0

Now, we require a binary blob to store the vertex and face information for our unit pipe.

In [8]:
# Empty array for appending binary blobs
main_binary_blob = bytearray()

Now, let's add our unit pipe object to our binary blob.

In [9]:
triangles_binary_blob = f_matrix.flatten().tobytes()
points_binary_blob = v_matrix.tobytes()

main_binary_blob = triangles_binary_blob + points_binary_blob

Now, we need to do some critical thinking. The transformation matrix will be saved at the node level. The nodes (however many there are), will reference a single mesh, which will reference a total of 2 bufferViews (one for the vertices an one for the indices). This baseline information should be standard for each gltf file, even before we iterate through our IFC geometry.

Hence, we set up the bufferviews as follows.

In [10]:
gltf_export.bufferViews.extend([
    pygltflib.BufferView(
        buffer=0,
        byteOffset=0,
        byteLength=len(triangles_binary_blob),
        target=pygltflib.ELEMENT_ARRAY_BUFFER
    ),
    pygltflib.BufferView(
        buffer=0,
        byteOffset=len(triangles_binary_blob),
        byteLength=len(points_binary_blob),
        target=pygltflib.ARRAY_BUFFER
    )
])

The attributes above bufferviews are Accessors, of which there should be 2 (once again, one for indices and one for vertices).

In [11]:
gltf_export.accessors.extend([
pygltflib.Accessor(
        bufferView=0,
        componentType=pygltflib.UNSIGNED_BYTE,
        count=f_matrix.size,
        type=pygltflib.SCALAR,
        max=[int(f_matrix.max())],
        min=[int(f_matrix.min())]
    ),
    pygltflib.Accessor(
        bufferView=1,
        componentType=pygltflib.FLOAT,
        count=len(v_matrix),
        type=pygltflib.VEC3,
        max=v_matrix.max(axis=0).tolist(),
        min=v_matrix.min(axis=0).tolist()
    )
])

Above accessors, we have meshes. There should only be 1 mesh here.

In [12]:
gltf_export.meshes.extend([
    pygltflib.Mesh(
        primitives=[
            pygltflib.Primitive(
                attributes=pygltflib.Attributes(POSITION=1), indices=0
            )
        ]
    )
])

And now, we can add the nodes. There will be a lot of nodes here- 1 for each unique mesh in the scene. As well, here we shall calculate the transformation matrix for each node in the scene. An important note is that the Transformation matrix must be saved as a flattened Column Major matrix. This means, we must transpose the matrix, then flatten it before we can save it as an attribute.

Here we also "import" our transformation matrix acquirer.

In [13]:
def get_transformation_matrix(input_v):
    '''
    Given a vertex array, finds the transformation matrix required to rotate, scale, and translate a shape into that position. Works best for pipes.

    inputs:
        input_v: numpy array

    returns:
        transformation_matrix: numpy array 
    '''
    # Step 1: Center vertices
    centroid = np.min(input_v, axis=0) + (np.max(input_v, axis=0) - np.min(input_v, axis=0))/2
    centered_v = input_v - centroid

    # Step 2: Find covariance matrix
    cov_mat = np.cov(centered_v, rowvar=False)

    # Step 3: Find eigenvectors and values of the cov matrix
    eg_val, eg_vec = np.linalg.eigh(cov_mat)
    eg_val = np.round(eg_val, 4)

    # Step 4: Reorder eigenvector matrix to match length axis
    length_axis = None
    minor_axis_1 = None
    minor_axis_2 = None

    if eg_val[0] == eg_val[1]: 
        length_axis = 2
        minor_axis_1 = 0
        minor_axis_2 = 1

    elif eg_val[0] == eg_val[2]:
        length_axis = 1
        minor_axis_1 = 0
        minor_axis_2 = 2

    elif eg_val[1] == eg_val[2]:
        length_axis = 0
        minor_axis_1 = 2
        minor_axis_2 = 1

    else:
        length_axis = np.nan
        minor_axis_1 = np.nan
        minor_axis_2 = np.nan

    eg_vec = eg_vec[:, [minor_axis_1, length_axis, minor_axis_2]]

    # Step 5: Find scale vector by projecting original verts onto the eg_vectors
    projected_v = centered_v @ eg_vec

    scale_vec = np.max(projected_v, axis = 0)

    # Step 6: Find the final transformation matrix
    T_matrix = np.eye(4, 4)
    T_matrix[:3, :3] = eg_vec @ np.diag(scale_vec)
    T_matrix[:3, 3] = centroid

    # Return

    return T_matrix

Lets construct our final loop.

In [14]:
# settings.set(settings.USE_WORLD_COORDS, True)
# ctr=0

# iterator = ifcopenshell.geom.iterator(
#     settings, model, include=pipes, geometry_library=geom_library)
# if iterator.initialize():
#     while True:
#         shape = iterator.get()
#         element = model.by_id(shape.id)

#         verts = ifcopenshell.util.shape.get_vertices(shape.geometry)

#         # Get the transformation matrix
#         T_matrix = get_transformation_matrix(verts)

#         flat_T_matrix = T_matrix.T.flatten().tolist()
#         # flat_T_matrix = [1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1]

#         gltf_export.nodes.extend([
#             pygltflib.Node(
#                 mesh=0,
#                 matrix=flat_T_matrix,
#                 name=f"mesh-{ctr}"
#             )
#         ])

#         gltf_export.scenes[0].nodes.extend([ctr])

#         ctr += 1

#         if not iterator.next():
#             break

#         if ctr == 10:
#             break

In the test case above, we limited to 10 pipes just to see if it worked properly. Lets assign the final buffer and export to see what we got.

In [15]:
# gltf_export.buffers.append(pygltflib.Buffer(byteLength=len(main_binary_blob)))
# gltf_export.set_binary_blob(main_binary_blob)

In [16]:
# filename = "test_mep_hyperinstance.glb"
# gltf_export.save(filename)

Opening In Blender, this is what we see.

![HyperInstance Run 1](../img/hyperinstance-export-1.png)

This looks very promising, and the objects are definitely looking like pipes. Let's extend this out to the entire scene.

## Extending to Full Scene

Now its time to apply this workflow to the full scene. We use the exact same loop from earlier, simply removing the early stopping condition. Let's see what we get. We should also remove the earlier implementation, since we'd likely get duplicated results or probably another error. Hence, we comment out the earlier code implementation and restart the kernel.

In [17]:
# settings.set(settings.USE_WORLD_COORDS, True)
# ctr=0

# iterator = ifcopenshell.geom.iterator(
#     settings, model, include=pipes, geometry_library=geom_library)

# if iterator.initialize():
#     while True:
#         shape = iterator.get()
#         element = model.by_id(shape.id)

#         verts = ifcopenshell.util.shape.get_vertices(shape.geometry)

#         # Get the transformation matrix
#         T_matrix = get_transformation_matrix(verts)

#         flat_T_matrix = T_matrix.T.flatten().tolist()

#         gltf_export.nodes.extend([
#             pygltflib.Node(
#                 mesh=0,
#                 matrix=flat_T_matrix,
#                 name=f"mesh-{ctr}"
#             )
#         ])

#         gltf_export.scenes[0].nodes.extend([ctr])

#         ctr += 1

#         if not iterator.next():
#             break

In [18]:
gltf_export.buffers.append(pygltflib.Buffer(byteLength=len(main_binary_blob)))
gltf_export.set_binary_blob(main_binary_blob)

In [19]:
filename = "sixty5-mep_hyperinstance.glb"
gltf_export.save(filename)

True

The first thing we notice is that our file size on disk is 2.5MB which is a stark reduction in size from our baseline. Opening this file in Blender, this is what we observe.

![Hyperinstance Run 2](../img/hyperinstance-export-2.png)

Oops. Looks like we might have applied an extra rotation somewhere. Not a worry, the overall shape of our model appears to be maintained. Let's load this to our testing threejs scene and see what we get.

![Hyperinstance-export-2-in scene](../img/hyperinstance-export-2-inscene.png)

While the orientation is definitely off, we see that our memory consumption peaks at only ~100MB, approximately 3X improvement over the baseline. Very promising results.

We need to apply a 90 degree clockwise rotation around the X axis to get our meshes in the correct orientation. Let's multiply our existing transformation matrix by another one signifying the rotation. For us, this matrix should be [[1,0,0], [0,0,1], [0,-1,0]]. Let's try this, once again, commenting out our eariler code.

In [20]:
settings.set(settings.USE_WORLD_COORDS, True)
ctr=0

iterator = ifcopenshell.geom.iterator(
    settings, model, include=pipes, geometry_library=geom_library)

if iterator.initialize():
    while True:
        shape = iterator.get()
        element = model.by_id(shape.id)

        verts = ifcopenshell.util.shape.get_vertices(shape.geometry)

        # Get the transformation matrix
        T_matrix = get_transformation_matrix(verts)

        add_90_deg_rotation = np.array([[1,0,0,0], [0,0,1,0], [0,-1,0,0], [0,0,0,1]])

        T_matrix = add_90_deg_rotation @ T_matrix

        flat_T_matrix = T_matrix.T.flatten().tolist()

        gltf_export.nodes.extend([
            pygltflib.Node(
                mesh=0,
                matrix=flat_T_matrix,
                name=f"mesh-{ctr}"
            )
        ])

        gltf_export.scenes[0].nodes.extend([ctr])

        ctr += 1

        if not iterator.next():
            break

In [21]:
gltf_export.buffers.append(pygltflib.Buffer(byteLength=len(main_binary_blob)))
gltf_export.set_binary_blob(main_binary_blob)

filename = "sixty5-mep_hyperinstance.glb"
gltf_export.save(filename)

True

An important consideration here is the order of operations. Matrix multiplication goes from right to left, hence to apply a global 90 degree transform, this matrix needs to be on the left of our existing transformation matrix.

And lets see what we get -->

![Hyperinstance Run 3](../img/hyperinstance-export-3.png)

Perfect. Our orientation is now aligned. Let's also load this to our three js testing environment and see what we get.

![Hyperinstance Run 3 in scene](../img/hyperinstance-export-3-inscene.png)

Perfect. Now, lets compare.

## Comparison of Results

| Model | Size on Disk | Peak RAM | Stable RAM | FPS |
| ----- | ------------ | -------- | ---------- | --- |
| Naive | 42 MB | 300MB | ~60MB | ~90 |
| Hyperinstanced | 2.5 MB | 100MB | 10.5 MB | ~96 |

In the table above, we measure performance results between our naive model and hyperinstanced one. 
- The naive model utilizes 42MB on disk, excluding any materials. 
- This model peaks at 300MB when loading the data to the scene. 
- Once the garbage collector is run, the scene uses approximately 60MB of stable RAM, at ~90FPS.

- Our hyperinstanced model utilizes only 2.5MB of space on the disk- **a 94% reduction in file size.**
- When loading to the scene, our RAM peaks at ~100MB- **a 66% reduction**.
- Once garbage collection is run, the stable RAM usage of the scene is ~10.5MB- **a ~82% improvement**

These are some fantastic results. We are strictly working with straight pipes here, but seeing a drastic improvement in the size of our file. Because the memory footprint of our hyperinstanced file is so low, we can theoretically even add a low-res LOD of our straight pipe to further improve the performance of the scene. Since the lowres model will be smaller in file size (due to there being less triangles), we could add both low and hi res versions to the same file, and still observe lower memory footprint than our naive model.